# 6BUIS017W CRM & CM with Business Intelligence – Individual Coursework (CWK1)
## Task 1: Data Preparation and Feature Engineering
**Market Analysis and Stock Segmentation Using RFM**

IIT ID: 20231076 | UoW ID: 20521398

**Static dataset:** the S&P 500 list changes over time, so the data was downloaded once (list retrieved on the date saved in `data/sp500_constituents.csv`) and saved in the repository. With `USE_SNAPSHOT = True` this notebook loads that saved data, so every run uses the same shares and prices. Set it to `False` to scrape Wikipedia and download from Yahoo Finance again.

## Setup


In [3]:
# Install the packages (versions are recorded for reproducibility)
!pip install -q yfinance==0.2.66 lxml

In [4]:
import os, sys, json, time
import datetime as dt
import numpy as np
import pandas as pd
import yfinance as yf
import matplotlib.pyplot as plt
import seaborn as sns

# ---- Study settings (from the coursework brief) ----
START = '2021-01-01'      # first date requested
END = '2025-01-01'        # end date is EXCLUSIVE (last possible trading day = 31 Dec 2024)
BENCH = '^GSPC'           # S&P 500 benchmark index
WINDOW = 60               # latest 60 trading sessions for Frequency and Monetary
BASELINE = 20             # preceding 20 sessions for the volume baseline
MULT = 2                  # high-volume day: volume > 2 x baseline mean

USE_SNAPSHOT = True       # True = load the saved static dataset, False = download again

GITHUB_USER = 'shahlyfayeek'
REPO = 'crm-cwk1-stock-segmentation'
SNAPSHOT_URL = 'https://raw.githubusercontent.com/' + GITHUB_USER + '/' + REPO + '/main/data/'

DATA_DIR = 'data'
FIG_DIR = 'figures'
os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(FIG_DIR, exist_ok=True)

# Package versions (required by the brief)
versions = {
    'python': sys.version.split()[0],
    'pandas': pd.__version__,
    'numpy': np.__version__,
    'yfinance': yf.__version__,
    'seaborn': sns.__version__,
    'matplotlib': __import__('matplotlib').__version__,
}
print(versions)

def snapshot_path(name):
    """Use the file in the Colab data folder if it is there, otherwise read it from the GitHub repo."""
    local = DATA_DIR + '/' + name
    if os.path.exists(local):
        return local
    return SNAPSHOT_URL + name

{'python': '3.13.18', 'pandas': '2.2.3', 'numpy': '2.4.6', 'yfinance': '0.2.66', 'seaborn': '0.13.2', 'matplotlib': '3.10.0'}


## 1.1 Ticker selection and data acquisition

**Choices**
* Ticker list: current S&P 500 constituents table from Wikipedia, read with `pd.read_html` (as in Week 4).
* Price source: **`yfinance`** (chosen over YahooFinancials because it was used in the Week 1–5 notebooks and downloads many tickers at once).
* Prices are downloaded with `auto_adjust=False` so I keep both `Close` (split-adjusted) and `Adj Close` (split- and dividend-adjusted) plus `Volume`.
* Study period: 1 Jan 2021 to 1 Jan 2025, end date exclusive.

In [5]:
# --- 1.1a  S&P 500 ticker list ---
if USE_SNAPSHOT:
    sp500 = pd.read_csv(snapshot_path('sp500_constituents.csv'))
    retrieval_date = sp500['Retrieval_Date'].iloc[0]
    print('Loaded the saved ticker list, retrieved on:', retrieval_date)
else:
    url = 'https://en.wikipedia.org/wiki/List_of_S%26P_500_companies'
    # A User-Agent is needed, otherwise Wikipedia can return HTTP 403 (Week 4 notebook)
    table = pd.read_html(url, storage_options={'User-Agent': 'Mozilla/5.0'})
    sp500 = table[0].copy()

    retrieval_date = dt.datetime.now(dt.timezone.utc).strftime('%Y-%m-%d')
    print('Constituent list retrieved on:', retrieval_date, '(UTC)')

    # Yahoo uses '-' instead of '.' (BRK.B -> BRK-B)
    sp500['Ticker'] = sp500['Symbol'].str.replace('.', '-', regex=False)
    sp500['Retrieval_Date'] = retrieval_date
    sp500 = sp500[['Ticker', 'Symbol', 'Security', 'GICS Sector', 'GICS Sub-Industry', 'Retrieval_Date']]

print('Number of tickers on the list:', len(sp500))
print('Duplicate tickers on the list:', sp500['Ticker'].duplicated().sum())
sp500.to_csv(DATA_DIR + '/sp500_constituents.csv', index=False)
sp500.head()

Loaded the saved ticker list, retrieved on: 2026-11-14
Number of tickers on the list: 503
Duplicate tickers on the list: 0


,Ticker,Symbol,Security,GICS Sector,GICS Sub-Industry,Retrieval_Date
0,MMM,MMM,3M,Industrials,Industrial Conglomerates,2026-11-14
1,AOS,AOS,A. O. Smith,Industrials,Building Products,2026-11-14
2,ABT,ABT,Abbott Laboratories,Health Care,Health Care Equipment,2026-11-14
3,ABBV,ABBV,AbbVie,Health Care,Biotechnology,2026-11-14
4,ACN,ACN,Accenture,Information Technology,IT Consulting & Other Services,2026-11-14


In [6]:
# --- 1.1b  Daily prices and volume (shares + benchmark) ---
tickers = sp500['Ticker'].tolist()
all_symbols = tickers + [BENCH]

if USE_SNAPSHOT:
    adj_raw = pd.read_csv(snapshot_path('raw_adj_close.csv.gz'), index_col=0, parse_dates=True)
    close_raw = pd.read_csv(snapshot_path('raw_close.csv.gz'), index_col=0, parse_dates=True)
    vol_raw = pd.read_csv(snapshot_path('raw_volume.csv.gz'), index_col=0, parse_dates=True)
    source_used = 'saved snapshot of the Yahoo Finance download (list retrieved ' + retrieval_date + ')'
else:
    # Download in chunks of 100 tickers to reduce Yahoo rate-limit problems
    parts = []
    for i in range(0, len(all_symbols), 100):
        chunk = all_symbols[i:i + 100]
        parts.append(yf.download(chunk, start=START, end=END, auto_adjust=False, progress=False))
        time.sleep(1)
    raw = pd.concat(parts, axis=1)
    adj_raw = raw['Adj Close']
    close_raw = raw['Close']
    vol_raw = raw['Volume']
    source_used = 'live Yahoo Finance download'

for table_ in (adj_raw, close_raw, vol_raw):
    table_.index = pd.to_datetime(table_.index)
    table_.sort_index(inplace=True)

print('Data source used:', source_used)
print('Adj Close shape (dates x symbols):', adj_raw.shape)
print('First date:', adj_raw.index.min().date(), '| Last date:', adj_raw.index.max().date())

Data source used: saved snapshot of the Yahoo Finance download (list retrieved 2026-11-14)
Adj Close shape (dates x symbols): (1005, 504)
First date: 2021-01-04 | Last date: 2024-12-31


In [7]:
# --- 1.1c  Save the snapshots (static dataset) ---
adj_raw.to_csv(DATA_DIR + '/raw_adj_close.csv.gz', compression='gzip')
close_raw.to_csv(DATA_DIR + '/raw_close.csv.gz', compression='gzip')
vol_raw.to_csv(DATA_DIR + '/raw_volume.csv.gz', compression='gzip')

# Which symbols have no data at all?
failed = [t for t in all_symbols if (t not in adj_raw.columns) or adj_raw[t].isna().all()]
print('Symbols with no data at all:', failed)
pd.Series(failed, name='Symbol', dtype=str).to_csv(DATA_DIR + '/download_failures.csv', index=False)

Symbols with no data at all: ['FDXF', 'HONA', 'Q', 'SNDK', 'VYLR']


## 1.2 Data validation and alignment

**Validation rules (a share is retained only if it passes all of them):**

| # | Rule | Why |
|---|---|---|
| R1 | Data downloaded | no data = nothing to analyse |
| R2 | Same history as the benchmark (first price on the first benchmark date) | unequal histories (new listings) give different numbers of weeks |
| R3 | No missing Adj Close, Close or Volume on any benchmark trading date | missing prices are **not filled** (the Week 4/5 `ffill/bfill` approach creates artificial zero returns and uses future prices) |
| R4 | All prices > 0 | non-positive prices are errors |
| R5 | Volume > 0 on every date | negative or zero volume for an S&P 500 share is a data error |

Duplicate dates and corporate actions (splits, dividends) are checked separately below.

In [8]:
# --- 1.2a  Benchmark and date checks ---
bench_adj = adj_raw[BENCH].dropna()
bench_dates = bench_adj.index
print('Benchmark trading days:', len(bench_dates))
print('Duplicate dates in the price table:', adj_raw.index.duplicated().sum())
print('Duplicate dates in the volume table:', vol_raw.index.duplicated().sum())
print('Dates outside the study period:', ((adj_raw.index < START) | (adj_raw.index >= END)).sum())
print('Dates not in the benchmark calendar:', len(adj_raw.index.difference(bench_dates)))
print('Benchmark non-positive prices:', (bench_adj <= 0).sum())

# Use the benchmark calendar for every share (identical dates)
adj = adj_raw.reindex(bench_dates)
close = close_raw.reindex(bench_dates)
vol = vol_raw.reindex(bench_dates)

Benchmark trading days: 1005
Duplicate dates in the price table: 0
Duplicate dates in the volume table: 0
Dates outside the study period: 0
Dates not in the benchmark calendar: 0
Benchmark non-positive prices: 0


In [9]:
# --- 1.2b  Apply the rules R1-R5 to every share and build the exclusion log ---
log_rows = []
for t in tickers:
    # R1 no data
    if (t not in adj.columns) or adj[t].isna().all():
        log_rows.append((t, 'R1 No data', 'No data downloaded'))
        continue

    # R2 unequal history: the first price is later than the first benchmark date
    first_valid = adj[t].first_valid_index()
    if first_valid > bench_dates[0]:
        log_rows.append((t, 'R2 Unequal history', 'First price on ' + str(first_valid.date())))

    # R3 missing values (only for shares with a full-length history)
    n_missing = int(adj[t].isna().sum() + close[t].isna().sum() + vol[t].isna().sum())
    if n_missing > 0 and first_valid == bench_dates[0]:
        log_rows.append((t, 'R3 Missing values', str(n_missing) + ' missing cells'))

    # R4 non-positive prices
    if (adj[t] <= 0).any() or (close[t] <= 0).any():
        log_rows.append((t, 'R4 Non-positive price', 'Adj Close or Close <= 0'))

    # R5 volume errors
    n_bad_volume = int((vol[t] <= 0).sum())
    if n_bad_volume > 0:
        log_rows.append((t, 'R5 Volume error', str(n_bad_volume) + ' day(s) with volume <= 0'))

exclusion_log = pd.DataFrame(log_rows, columns=['Ticker', 'Rule', 'Detail'])
excluded = set(exclusion_log['Ticker'])
included = [t for t in tickers if t not in excluded]

print('Tickers on the list      :', len(tickers))
print('Tickers excluded         :', len(excluded))
print('Tickers retained         :', len(included))
print()
print('Exclusions by rule:')
print(exclusion_log['Rule'].value_counts())

exclusion_log.to_csv(DATA_DIR + '/exclusion_log.csv', index=False)
pd.Series(included, name='Ticker').to_csv(DATA_DIR + '/included_tickers.csv', index=False)
exclusion_log

Tickers on the list      : 503
Tickers excluded         : 17
Tickers retained         : 486

Exclusions by rule:
Rule
R2 Unequal history    10
R1 No data             5
R5 Volume error        2
Name: count, dtype: int64


,Ticker,Rule,Detail
0,APP,R2 Unequal history,First price on 2021-04-15
1,COIN,R2 Unequal history,First price on 2021-04-14
2,CEG,R2 Unequal history,First price on 2022-01-19
3,EXE,R2 Unequal history,First price on 2021-02-10
4,FDXF,R1 No data,No data downloaded
5,FERG,R5 Volume error,27 day(s) with volume <= 0
6,GEHC,R2 Unequal history,First price on 2022-12-15
7,GEV,R2 Unequal history,First price on 2024-03-27
8,HONA,R1 No data,No data downloaded
9,Q,R1 No data,No data downloaded


In [10]:
# --- 1.2c  Corporate action checks ---
adj_in = adj[included]
close_in = close[included]
vol_in = vol[included]

# (i) Shares whose Adj Close is below Close at the start: dividends are being adjusted
ratio_first = adj_in.iloc[0] / close_in.iloc[0]
print('Shares where Adj Close < Close at the start (dividend adjustment):',
      int((ratio_first < 0.9999).sum()), 'of', len(included))

# (ii) Very large one-day moves in Close can mean an unadjusted split
close_ret = close_in.pct_change(fill_method=None)
adj_ret = adj_in.pct_change(fill_method=None)
flag_rows = []
for t in included:
    big_days = close_ret.index[close_ret[t].abs() > 0.35]
    for d in big_days:
        flag_rows.append((t, d.date(), round(close_ret.loc[d, t], 3), round(adj_ret.loc[d, t], 3)))
flag_table = pd.DataFrame(flag_rows, columns=['Ticker', 'Date', 'Close_return', 'AdjClose_return'])
print('Days with an absolute daily Close move above 35%:', len(flag_table))
flag_table.to_csv(DATA_DIR + '/corporate_action_flags.csv', index=False)
flag_table

Shares where Adj Close < Close at the start (dividend adjustment): 398 of 486
Days with an absolute daily Close move above 35%: 15


,Ticker,Date,Close_return,AdjClose_return
0,BIIB,2021-06-07,0.383,0.383
1,BIIB,2022-09-28,0.399,0.399
2,BE,2021-10-25,0.371,0.371
3,BE,2024-11-15,0.592,0.592
4,CVNA,2022-08-05,0.401,0.401
5,CVNA,2022-11-04,-0.390,-0.390
6,CVNA,2022-12-07,-0.429,-0.429
7,CVNA,2023-01-12,0.460,0.460
8,CVNA,2023-06-08,0.560,0.560
9,CVNA,2023-07-19,0.402,0.402


In [11]:
# (iii) Check Yahoo's recorded stock splits for the flagged shares only
split_rows = []
for t in sorted(flag_table['Ticker'].unique()):
    try:
        s = yf.Ticker(t).splits
        s.index = pd.to_datetime(s.index).tz_localize(None)
        s = s[(s.index >= START) & (s.index < END)]
        for d, ratio in s.items():
            split_rows.append((t, d.date(), ratio))
    except Exception as e:
        print(t, 'split lookup failed:', e)

split_table = pd.DataFrame(split_rows, columns=['Ticker', 'Split_date', 'Split_ratio'])
split_table.to_csv(DATA_DIR + '/splits_found.csv', index=False)
print('Splits found for flagged shares in the period:')
split_table

Splits found for flagged shares in the period:


,Ticker,Split_date,Split_ratio
0,DXCM,2022-06-13,4.0
1,SMCI,2024-10-01,10.0


**Corporate actions**
Splits and dividends change a share's price without changing the value of the company, so I checked that the prices were adjusted for them. Adj Close is adjusted for both and Close for splits only. 398 of 486 shares have Adj Close below Close at the start, which shows the dividends are included. I looked for one-day moves above 35% because an unadjusted split would show up like that. There were 15, but Close and Adj Close gave the same return on each, and none fell on the two split dates Yahoo gave (DXCM and SMCI). So they are real price moves, not data problems. I use Adj Close for returns and beta, and Close x Volume for Monetary. The benchmark is a price index without dividends, so its returns are slightly lower than a total return, which is a small limitation for beta.

**Exclusions**
17 of 503 tickers were excluded: 5 had no data (R1), 10 listed after January 2021 so have a shorter history (R2), and 2 (FERG and SW) had days with zero volume (R5). I did not fill any missing prices.

**Survivorship bias**
The share list is today's S&P 500 but I use it for 2021-2024. Companies that were removed from the index in that time are missing, and companies that joined later are included. This favours winners, so average returns look higher and risk lower than they really were.

## 1.3.1 Mean weekly return
* Friday-ending weeks: `resample('W-FRI').last()` (final available trading price of the week, so holiday-shortened weeks are kept).
* Weekly return = adjusted close at the end of this week / adjusted close at the end of the previous week - 1.
* **Incomplete first and final calendar weeks are excluded.** A week is kept only if its Monday is on or after the start date and its Friday is before the exclusive end date.
* Mean weekly return = arithmetic mean over the full period.

In [12]:
# --- Weekly prices for shares and benchmark ---
adj_week = adj_in.resample('W-FRI').last()
bench_week = bench_adj.resample('W-FRI').last()

# A week is complete if its Monday is on or after START and its Friday is before END
week_monday = adj_week.index - pd.Timedelta(days=4)
complete = (week_monday >= pd.Timestamp(START)) & (adj_week.index < pd.Timestamp(END))
print('Weeks removed as incomplete:', [d.date() for d in adj_week.index[~complete]])

adj_week = adj_week[complete]
bench_week = bench_week[complete]
print('First complete week:', adj_week.index[0].date(), '| Last complete week:', adj_week.index[-1].date())
print('Number of complete weekly prices:', len(adj_week))

# Weekly returns (no filling: fill_method=None). The first week has no previous week.
weekly_ret = adj_week.pct_change(fill_method=None).iloc[1:]
bench_ret = bench_week.pct_change(fill_method=None).iloc[1:]

# Common week labels
print('Identical week labels for shares and benchmark:', weekly_ret.index.equals(bench_ret.index))
print('Missing weekly returns in the share table:', int(weekly_ret.isna().sum().sum()))
print('Missing weekly returns for the benchmark:', int(bench_ret.isna().sum()))
print('Number of weekly return observations:', len(bench_ret))

Weeks removed as incomplete: [datetime.date(2025, 1, 3)]
First complete week: 2021-01-08 | Last complete week: 2024-12-27
Number of complete weekly prices: 208
Identical week labels for shares and benchmark: True
Missing weekly returns in the share table: 0
Missing weekly returns for the benchmark: 0
Number of weekly return observations: 207


In [13]:
# --- Mean weekly return (arithmetic mean) ---
mean_ret = pd.DataFrame({'Mean_Weekly_Return': weekly_ret.mean(),
                         'Aligned_Weeks': weekly_ret.notna().sum()})
mean_ret.index.name = 'Ticker'
print('Aligned weekly observations per share (unique values):', mean_ret['Aligned_Weeks'].unique())

mean_ret.to_csv(DATA_DIR + '/mean_weekly_return.csv')

weekly_all = weekly_ret.copy()
weekly_all[BENCH] = bench_ret
weekly_all.to_csv(DATA_DIR + '/weekly_returns.csv.gz', compression='gzip')
mean_ret.head()

Aligned weekly observations per share (unique values): [207]


,Mean_Weekly_Return,Aligned_Weeks
Ticker,,
MMM,0.001142,207
AOS,0.001991,207
ABT,0.000971,207
ABBV,0.003749,207
ACN,0.002376,207


In [14]:
# --- Check the return formula and the mean by hand on one share ---
check = included[0]
print('Check share:', check)

prices = adj_week[check].tail(4)
by_hand = (prices / prices.shift(1) - 1).dropna()
print('By-hand weekly returns:')
print(by_hand.round(5).to_string())
print('Table values:')
print(weekly_ret[check].tail(3).round(5).to_string())
assert np.allclose(by_hand.values, weekly_ret[check].tail(3).values)

print('Mean weekly return, by hand :', round(weekly_ret[check].sum() / len(weekly_ret), 6))
print('Mean weekly return, table   :', round(mean_ret.loc[check, 'Mean_Weekly_Return'], 6))

Check share: MMM
By-hand weekly returns:
Date
2024-12-13   -0.02397
2024-12-20   -0.00493
2024-12-27    0.00696
Freq: W-FRI
Table values:
Date
2024-12-13   -0.02397
2024-12-20   -0.00493
2024-12-27    0.00696
Freq: W-FRI
Mean weekly return, by hand : 0.001142
Mean weekly return, table   : 0.001142


## 1.3.2 Weekly beta
Weekly beta = covariance(share weekly returns, benchmark weekly returns) / variance(benchmark weekly returns), using the full study period on matched week labels (both use the sample formula, ddof = 1). No missing weeks are filled and no multi-week change is treated as a one-week return.

In [15]:
beta_list = []
weeks_list = []
for t in included:
    share = weekly_ret[t]
    beta = share.cov(bench_ret) / bench_ret.var()        # cov() lines the two series up by week label
    n_weeks = int((share.notna() & bench_ret.notna()).sum())
    beta_list.append(beta)
    weeks_list.append(n_weeks)

beta_df = pd.DataFrame({'Weekly_Beta': beta_list, 'Aligned_Weeks': weeks_list}, index=included)
beta_df.index.name = 'Ticker'
print('Aligned weekly observations used for beta:', beta_df['Aligned_Weeks'].unique())
print('Benchmark weekly return variance:', round(bench_ret.var(), 8))
beta_df.to_csv(DATA_DIR + '/weekly_beta.csv')
beta_df.head()

Aligned weekly observations used for beta: [207]
Benchmark weekly return variance: 0.00049218


,Weekly_Beta,Aligned_Weeks
Ticker,,
MMM,0.869079,207
AOS,1.042805,207
ABT,0.583543,207
ABBV,0.425366,207
ACN,1.143299,207


In [16]:
# --- Check: beta from the slope of a fitted straight line ---
check = included[0]
slope = np.polyfit(bench_ret.values, weekly_ret[check].values, 1)[0]
print('Check share:', check)
print('Beta (covariance / variance):', round(beta_df.loc[check, 'Weekly_Beta'], 6))
print('Beta (regression slope)     :', round(slope, 6))
assert abs(slope - beta_df.loc[check, 'Weekly_Beta']) < 1e-8

Check share: MMM
Beta (covariance / variance): 0.869079
Beta (regression slope)     : 0.869079


## 1.4 Adapted RFM calculations

RFM is adapted from customer purchasing to **share trading activity**. It does not measure company value, loyalty or profitability.

* **Observation date** = final common completed trading date of the study period.
* **High-volume day:** volume > 2 x the mean volume of the **preceding 20 trading sessions, excluding the day being tested** (`shift(1)` then `rolling(20).mean()`). The threshold is calculated separately for every share and every day.
* **Recency (R):** calendar days from the latest high-volume day (in the valid history) to the observation date. Same day = 0. If there is no event, R is left missing (not 0).
* **Frequency (F):** number of high-volume days in the latest 60 sessions ending on the observation date.
* **Monetary (M):** sum of `Close x Volume` over those 60 sessions (USD, approximate traded value).

In [17]:
# Observation date = last date on which every retained share has data
obs_date = close_in.dropna(how='any').index.max()
print('Observation date:', obs_date.date())

# Use data up to the observation date only
vol_use = vol_in.loc[:obs_date]
close_use = close_in.loc[:obs_date]

# Mean volume of the previous 20 sessions (shift(1) leaves the current day out), for every share and day
baseline = vol_use.shift(1).rolling(BASELINE).mean()
threshold = MULT * baseline

# A high-volume day: volume is greater than the threshold (days without a full baseline are never events)
high_volume = vol_use > threshold

# History check: 20 sessions of baseline for each of the 60 window days = 80 sessions
sessions = vol_use.notna().sum()
print('Shares with at least', WINDOW + BASELINE, 'sessions of history:', int((sessions >= WINDOW + BASELINE).sum()), 'of', len(included))

Observation date: 2024-12-31
Shares with at least 80 sessions of history: 486 of 486


### 1.4.1 Recency

In [18]:
recency_list = []
last_event_list = []
for t in included:
    event_days = high_volume.index[high_volume[t]]          # all high-volume days of this share
    if len(event_days) == 0:
        recency_list.append(np.nan)                          # no event: missing, not 0
        last_event_list.append(None)
    else:
        recency_list.append((obs_date - event_days[-1]).days)
        last_event_list.append(event_days[-1].date())

recency = pd.DataFrame({'Recency': recency_list, 'Latest_Event_Date': last_event_list}, index=included)
recency.index.name = 'Ticker'
print('Shares with R missing (no high-volume event):', int(recency['Recency'].isna().sum()))

# Shares with no event cannot go into RFM clustering: log them
no_event = recency[recency['Recency'].isna()]
if len(no_event) > 0:
    extra = pd.DataFrame({'Ticker': no_event.index,
                          'Rule': 'RFM: no event',
                          'Detail': 'No high-volume day in valid history, R missing, excluded from RFM clustering (not set to 0)'})
    exclusion_log = pd.concat([exclusion_log, extra], ignore_index=True)
    exclusion_log.to_csv(DATA_DIR + '/exclusion_log.csv', index=False)

recency.to_csv(DATA_DIR + '/recency.csv')
recency.head()

Shares with R missing (no high-volume event): 0


,Recency,Latest_Event_Date
Ticker,,
MMM,11,2024-12-20
AOS,11,2024-12-20
ABT,11,2024-12-20
ABBV,11,2024-12-20
ACN,11,2024-12-20


### 1.4.2 Frequency

In [19]:
# Count the high-volume days in the latest 60 sessions
frequency = pd.DataFrame({'Frequency': high_volume.tail(WINDOW).sum()})
frequency.index.name = 'Ticker'
print(frequency['Frequency'].describe().round(2))
frequency.to_csv(DATA_DIR + '/frequency.csv')
frequency.head()

count    486.00
mean       2.99
std        1.33
min        0.00
25%        2.00
50%        3.00
75%        4.00
max        8.00
Name: Frequency, dtype: float64


,Frequency
Ticker,
MMM,4
AOS,2
ABT,1
ABBV,3
ACN,2


### 1.4.3 Monetary

In [20]:
# Sum of Close x Volume over the latest 60 sessions (USD)
monetary = pd.DataFrame({'Monetary': (close_use.tail(WINDOW) * vol_use.tail(WINDOW)).sum()})
monetary.index.name = 'Ticker'
print('Monetary is in USD (sum of Close x Volume over 60 sessions)')
print(monetary['Monetary'].describe().apply(lambda x: '{:,.0f}'.format(x)))
monetary.to_csv(DATA_DIR + '/monetary.csv')
monetary.head()

Monetary is in USD (sum of Close x Volume over 60 sessions)
count                  486
mean        39,404,191,889
std        129,138,833,233
min          1,297,482,617
25%         10,195,173,392
50%         16,683,821,505
75%         32,276,893,044
max      1,847,697,379,222
Name: Monetary, dtype: object


,Monetary
Ticker,
MMM,2.599146e+10
AOS,4.748980e+09
ABT,3.523027e+10
ABBV,6.837054e+10
ACN,5.199430e+10


## 1.5 Metric interpretation and supporting evidence

### Worked example and checks
The first cell shows every RFM step for one share. The second cell checks the threshold alignment and tests the rules on small made-up series (the numbers from the brief), including the boundary cases.

In [21]:
# --- Worked example for one share ---
example = recency['Recency'].dropna().index[0]
lines = []

example_table = pd.DataFrame({'Close': close_use[example],
                              'Volume': vol_use[example],
                              'Mean_prev_20': baseline[example].round(0),
                              'Threshold_2x': threshold[example].round(0),
                              'High_volume_day': high_volume[example]}).tail(8)
print(example_table.to_string())
example_table.to_csv(DATA_DIR + '/worked_example.csv')

lines.append('Worked example share: ' + example + ' | Observation date: ' + str(obs_date.date()))

# Recency by hand
event_days = high_volume.index[high_volume[example]]
last_event = event_days[-1]
recency_hand = (obs_date - last_event).days
lines.append('Latest high-volume day: ' + str(last_event.date()))
lines.append('Recency = observation date - event date = ' + str(recency_hand) + ' calendar days')

# Frequency by hand (latest 60 sessions)
window_flags = high_volume[example].tail(WINDOW)
frequency_hand = int(window_flags.sum())
lines.append('60-session window runs from ' + str(window_flags.index[0].date()) + ' to ' + str(window_flags.index[-1].date()))
lines.append('Frequency = high-volume days in the window = ' + str(frequency_hand))

# Monetary by hand
daily_value = close_use[example].tail(WINDOW) * vol_use[example].tail(WINDOW)
monetary_hand = daily_value.sum()
lines.append('Monetary = sum of 60 daily Close x Volume = USD {:,.0f}'.format(monetary_hand))

# The by-hand numbers must match the tables from sections 1.4.1 - 1.4.3
assert recency_hand == recency.loc[example, 'Recency']
assert frequency_hand == frequency.loc[example, 'Frequency']
assert abs(monetary_hand - monetary.loc[example, 'Monetary']) < 1
lines.append('By-hand R, F and M match the tables above.')
print()
print('\n'.join(lines))

                 Close   Volume  Mean_prev_20  Threshold_2x  High_volume_day
Date                                                                        
2024-12-19  127.129997  3038100     2960665.0     5921330.0            False
2024-12-20  129.279999  8351300     3027540.0     6055080.0             True
2024-12-23  128.979996  2067400     3317865.0     6635730.0            False
2024-12-24  130.360001   803200     3268615.0     6537230.0            False
2024-12-26  131.179993  1484900     3055040.0     6110080.0            False
2024-12-27  130.179993  1842600     2975270.0     5950540.0            False
2024-12-30  129.130005  2154000     2938420.0     5876840.0            False
2024-12-31  129.089996  2220300     2965205.0     5930410.0            False

Worked example share: MMM | Observation date: 2024-12-31
Latest high-volume day: 2024-12-20
Recency = observation date - event date = 11 calendar days
60-session window runs from 2024-10-07 to 2024-12-31
Frequency = high-volume d

In [22]:
# --- Check 1: threshold alignment (the baseline uses the 20 sessions BEFORE the day) ---
v = vol_use[example]
pos = len(v) - 10                                   # test a day 10 sessions before the end
manual_baseline = v.iloc[pos - BASELINE:pos].mean() # sessions pos-20 ... pos-1 (current day excluded)
lines.append('Baseline from rolling code: ' + str(round(baseline[example].iloc[pos], 2)) + ' | by hand: ' + str(round(manual_baseline, 2)))
assert abs(baseline[example].iloc[pos] - manual_baseline) < 1e-6

# --- Check 2: made-up data (100 trading days) ---
days = pd.bdate_range('2024-01-01', periods=100)
price = pd.Series(100.0, index=days)

# (a) the brief's numbers: baseline about 1 million shares
vol_a = pd.Series(1_000_000.0, index=days)
vol_a.iloc[85] = 2_000_000    # exactly 2 x baseline -> NOT an event (it must be greater)
vol_a.iloc[90] = 1_800_000    # below the threshold -> NOT an event
vol_a.iloc[95] = 2_500_000    # above the threshold -> event
event_a = vol_a > 2 * vol_a.shift(1).rolling(BASELINE).mean()
assert (not event_a.iloc[85]) and (not event_a.iloc[90]) and event_a.iloc[95]
assert event_a.tail(WINDOW).sum() == 1
assert (days[-1] - days[event_a][-1]).days == (days[-1] - days[95]).days
lines.append('Test (a): 2.0m and 1.8m are not events, 2.5m is. F = 1, R = ' + str((days[-1] - days[95]).days))

# (b) an event on the observation date gives R = 0
vol_b = pd.Series(1_000_000.0, index=days)
vol_b.iloc[-1] = 3_000_000
event_b = vol_b > 2 * vol_b.shift(1).rolling(BASELINE).mean()
assert (days[-1] - days[event_b][-1]).days == 0
lines.append('Test (b): an event on the observation date gives R = 0')

# (c) no event: nothing is flagged, so R stays missing (not 0)
vol_c = pd.Series(1_000_000.0, index=days)
event_c = vol_c > 2 * vol_c.shift(1).rolling(BASELINE).mean()
assert event_c.sum() == 0
lines.append('Test (c): no event gives F = 0 and R missing')

# (d) Monetary: 100 USD x 2 million shares = 200 million a day, x 60 days = 12 billion
vol_d = pd.Series(2_000_000.0, index=days)
monetary_d = (price.tail(WINDOW) * vol_d.tail(WINDOW)).sum()
assert monetary_d == 60 * 200_000_000
lines.append('Test (d): Monetary = USD {:,.0f}'.format(monetary_d))

# (e) history: 60 window days each need 20 earlier sessions
lines.append('Test (e): at least ' + str(WINDOW + BASELINE) + ' sessions are needed; all retained shares have ' + str(int(sessions.min())) + ' or more')
assert sessions.min() >= WINDOW + BASELINE

print('\n'.join(lines[-6:]))
with open(DATA_DIR + '/validation_checks.txt', 'w') as f:
    f.write('\n'.join(lines))

Baseline from rolling code: 2890235.0 | by hand: 2890235.0
Test (a): 2.0m and 1.8m are not events, 2.5m is. F = 1, R = 4
Test (b): an event on the observation date gives R = 0
Test (c): no event gives F = 0 and R missing
Test (d): Monetary = USD 12,000,000,000
Test (e): at least 80 sessions are needed; all retained shares have 1005 or more
